# **1. Metodologia**

Este notebook transforma as disciplinas únicas em medidas e arquivos para a visualização.

| pergunta | medida principal | visão |
|---|---|---|
| Q1 · agrupamentos | BERTopic e rede de vizinhanças | mapa em rede |
| Q2 · semelhança a uma referência | cosseno entre embeddings | beeswarm por tópico |
| Q3 · encadeamento | grafo de requisitos por curso | grade curricular |

A carga horária por tópico também é calculada e exportada para a composição dos cursos, prevista para a próxima entrega.

Entradas: `data/processed/subjects.csv` e `subject_requisites.csv`. Saída web: `html/data/disciplinas.js`.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from hdbscan import HDBSCAN
from IPython.display import display
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import pdist
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score
from umap import UMAP

### **1.1. Parâmetros e entradas**

O bloco define modelo, semente, tamanho mínimo dos tópicos e regras da rede. Em seguida carrega as tabelas processadas e a auditoria da coleta.

In [2]:
warnings.filterwarnings("ignore")
os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for noisy in ("BERTopic", "torch.distributed.elastic.multiprocessing.redirects"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DATA = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
SITE_DATA = ROOT / "html" / "data"

MODEL = "BAAI/bge-m3"
SEED = 7
MIN_TOPIC = 8
K = 4
FLOOR_PERCENTILE = 85
REFRESH_EMBEDDINGS = False

subjects = pd.read_csv(PROCESSED / "subjects.csv", dtype={"codes": str}).fillna({"teachers": "", "ementa": "", "objectives": "", "program": ""})
pairs = pd.read_csv(PROCESSED / "subject_requisites.csv")
disciplines = pd.read_csv(PROCESSED / "disciplines.csv", dtype=str).fillna("")
collection = pd.read_csv(RAW_DATA / "collection_audit.csv").set_index("metric")["value"]
n = len(subjects)
print(f"{n} disciplinas únicas, {len(pairs)} pares de requisito")

528 disciplinas únicas, 702 pares de requisito


## **2. Embeddings**

O modelo `BAAI/bge-m3` representa cada texto por um vetor de 1.024 dimensões e lê até 8.192 tokens, o suficiente para o texto completo de todas as disciplinas, sem divisão em blocos. Somente o texto de ementa, objetivos e conteúdo programático é representado, e o nome não entra na análise. Se $f(x_d)$ é a saída do modelo para o texto $x_d$, o vetor da disciplina é

$$
\mathbf{e}_d=\frac{f(x_d)}{\lVert f(x_d)\rVert}.
$$

O cache fica em `data/processed/embeddings.npz` e é refeito quando mudam modelo ou textos, e `REFRESH_EMBEDDINGS = True` força o recálculo.

In [3]:
signature = hashlib.sha1("\n".join([MODEL, *subjects["subject_id"], *subjects["body_all"]]).encode()).hexdigest()
cache = PROCESSED / "embeddings.npz"
stored = np.load(cache, allow_pickle=True) if cache.exists() else None
if stored is not None and not REFRESH_EMBEDDINGS and "signature" in stored and str(stored["signature"]) == signature:
    E, longest = stored["vectors"], int(stored["longest"])
else:
    model = SentenceTransformer(MODEL)
    texts = subjects["body_all"].tolist()
    longest = max(len(model.tokenizer(text)["input_ids"]) for text in texts)
    assert longest <= model.max_seq_length
    E = model.encode(texts, normalize_embeddings=True, batch_size=8, show_progress_bar=False).astype(np.float32)
    np.savez(cache, ids=subjects["subject_id"].to_numpy(), vectors=E, signature=signature, longest=longest)
print(f"embeddings: {E.shape} · maior texto: {longest} tokens (limite do modelo: 8.192)")

embeddings: (528, 1024) · maior texto: 1523 tokens (limite do modelo: 8.192)


## **3. Semelhança**

A semelhança é o cosseno entre embeddings normalizados:

$$
s(a,b)=\frac{\mathbf{e}_a\cdot\mathbf{e}_b}{\lVert\mathbf{e}_a\rVert\,\lVert\mathbf{e}_b\rVert}.
$$

A distribuição de todos os pares fornece a referência para interpretar vizinhanças e requisitos. Os limiares são definidos por percentis dessa distribuição, e não por valores fixos, porque a escala do cosseno depende do modelo. O percentil 85 é o limiar da rede, e o percentil 90 é o limiar da Q2: uma disciplina só entra na comparação se for mais parecida com a referência do que 90% dos pares do conjunto.

In [4]:
S = E @ E.T
np.fill_diagonal(S, -1)
iu = np.triu_indices(n, 1)
all_pairs = S[iu]
summary = pd.Series(all_pairs).describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3)
display(summary.to_frame("todos os pares").T)
FLOOR = float(np.percentile(all_pairs, FLOOR_PERCENTILE))
P90 = float(np.percentile(all_pairs, 90))
print(f"rede · limiar {FLOOR:.3f} (percentil {FLOOR_PERCENTILE})")
above = (S >= P90).sum(axis=1)
print(f"Q2 · limiar {P90:.3f}: mediana de {np.median(above):.0f} disciplinas por referência, "
      f"{int((above < 5).sum())} referências têm menos de cinco")

top = np.argsort(-all_pairs)[:8]
pd.DataFrame([{"a": subjects.at[iu[0][k], "name"], "b": subjects.at[iu[1][k], "name"], "cosseno": round(float(all_pairs[k]), 3)} for k in top])

,count,mean,std,min,10%,25%,50%,75%,90%,99%,max
todos os pares,139128.0,0.543,0.062,0.265,0.467,0.502,0.54,0.581,0.619,0.702,1.0


rede · limiar 0.603 (percentil 85)
Q2 · limiar 0.619: mediana de 43 disciplinas por referência, 10 referências têm menos de cinco


,a,b,cosseno
0,Atividades de Extensão para Licenciatura 3,Atividades de Extensão para Licenciatura 4,1.0
1,Atividades de Extensão para Engenharia II,Atividades de Extensão para Licenciatura 3,1.0
2,Atividades de Extensão para Engenharia I,Atividades de Extensão para Licenciatura 4,1.0
3,Atividades de Extensão para Engenharia I,Atividades de Extensão para Licenciatura 3,1.0
4,Atividades de Extensão para Engenharia I,Atividades de Extensão para Engenharia II,1.0
5,Atividades de Extensão para Engenharia II,Atividades de Extensão para Licenciatura 4,1.0
6,Projeto Empreendedor I,Projeto Empreendedor II,1.0
7,Projeto em Intercâmbio I,Projeto em Intercâmbio II,1.0


## **4. Tópicos com BERTopic**

O agrupamento segue quatro passos.

| passo | método | função |
|---:|---|---|
| 1 | UMAP | reduzir os embeddings a 5 dimensões |
| 2 | HDBSCAN | encontrar grupos com pelo menos 8 disciplinas |
| 3 | c-TF-IDF | destacar palavras específicas de cada grupo |
| 4 | MMR | reduzir palavras redundantes |

Os documentos são os textos de ementa, objetivos e conteúdo programático, sem o nome. Palavras vazias e termos genéricos de ementa são removidos. Itens sem grupo recebem o tópico de centróide mais próximo.

In [5]:
STOP = sorted(set(unicodedata.normalize("NFC", """
a à ao aos as às o os um uma uns umas de da do das dos dum duma em na no nas nos num numa por pela pelo pelas pelos
para pra com sem sob sobre entre até após contra desde perante e ou nem mas porém que se como quando onde quanto
qual quais cujo cuja mais menos muito muitos muita muitas pouco pouca tão tanto também já ainda sempre não sim
seu sua seus suas este esta estes estas esse essa esses essas isso isto aquele aquela aqueles aquelas
ele ela eles elas lhe lhes cada outro outra outros outras mesmo mesma mesmos mesmas todo toda todos todas tal tais
ser é são foi era será serão sendo sido estar está estão ter tem têm haver há pode podem deve devem fazer feito
através via além bem assim etc i ii iii iv v vi
disciplina disciplinas curso cursos aula aulas aluno alunos aluna estudante estudantes professor objetivo objetivos
conceito conceitos básico básicos básica básicas introdução introdutório introdutória fundamentos fundamental
principais principal diversos diversas noções noção visão geral estudo estudos apresentar apresentação abordar
abordagem uso utilização utilizando usando forma formas tipo tipos aspecto aspectos tópico tópicos tema temas
desenvolvimento desenvolver capacidade capaz habilidades habilidade conhecimento conhecimentos compreender
entender permitir oferecer fornecer área áreas parte partes caso casos exemplo exemplos nível níveis
seguintes diferentes relacionados relacionadas importância dentro sistema
""").split()))

docs = subjects["body_all"].tolist()
vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2, stop_words=STOP, token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
mmr = MaximalMarginalRelevance(diversity=0.3, top_n_words=30)
topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=12, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=MIN_TOPIC, min_samples=3, prediction_data=True),
    vectorizer_model=vectorizer, representation_model=mmr, top_n_words=30,
    language="multilingual", calculate_probabilities=False, verbose=False,
)
raw_topics, _ = topic_model.fit_transform(docs, E)
logging.getLogger("BERTopic").setLevel(logging.ERROR)
noise = int(sum(t == -1 for t in raw_topics))
raw_topics = topic_model.reduce_outliers(docs, raw_topics, strategy="embeddings", embeddings=E)
topic_model.update_topics(docs, topics=raw_topics, vectorizer_model=vectorizer, representation_model=mmr, top_n_words=30)

order = [t for t, _ in Counter(raw_topics).most_common()]
topic = np.array([order.index(t) for t in raw_topics])
T = len(order)
words = [[(w, float(v)) for w, v in topic_model.get_topic(t)][:30] for t in order]
keys = [" ".join(w for w, _ in words[t][:2]) for t in range(T)]
print(f"{T} tópicos, {noise} disciplinas sem tópico no HDBSCAN foram reatribuídas")
pd.DataFrame({"tópico": range(T), "chave": keys, "disciplinas": [int((topic == t).sum()) for t in range(T)],
              "palavras": [", ".join(w for w, _ in words[t][:8]) for t in range(T)],
              "exemplos": [" | ".join(subjects.loc[topic == t, "name"].head(4)) for t in range(T)]})

24 tópicos, 74 disciplinas sem tópico no HDBSCAN foram reatribuídas


,tópico,chave,disciplinas,palavras,exemplos
0,0,algoritmos aprendizado,37,"algoritmos, aprendizado, árvores, busca, objet...",Introdução à Ciência de Computação II | Algori...
1,1,sistemas controle,35,"sistemas, controle, robôs, embarcados, navegaç...",Introdução à Engenharia Biomédica | Robótica E...
2,2,programas linguagens,35,"programas, linguagens, programação, computação...",Teoria da Computação e Linguagens Formais | Co...
3,3,software requisitos,33,"software, requisitos, gerenciamento, sistemas,...",Seminários em Gestão Organizacional | Comércio...
4,4,funções equações,33,"funções, equações, séries, números, teorema, m...",Matemática Aplicada à Química | Introdução à F...
5,5,espaços lineares,32,"espaços, lineares, teorema, matrizes, grupos, ...",Introdução à Geometria Computacional: Algoritm...
6,6,modelos testes,32,"modelos, testes, regressão, estatística, variá...",Controle Adaptativo | Processos Estocásticos |...
7,7,circuitos corrente,30,"circuitos, corrente, elétricos, transistores, ...",Física IV | Física III | Física Aplicada para ...
8,8,momento mecânica,28,"momento, mecânica, movimento, energia, calor, ...",Anatomia e Fisiologia para o Engenheiro Bioméd...
9,9,química reações,28,"química, reações, ensino, estágio, escola, quí...",Química Geral | Química Geral Experimental | G...


## **5. Nomes dos tópicos e dicionário de palavras**

O BERTopic entrega palavras, não nomes. Cada tópico recebe um nome curto a partir das palavras de maior peso e das disciplinas do grupo. `data/processed/topic_dictionary.csv` preserva palavras, pesos e posições.

A ordem dos tópicos na visualização (e, com ela, a ordem das cores) segue as folhas de um dendrograma de ligação média sobre os centróides dos tópicos: tópicos parecidos ficam lado a lado e recebem matizes vizinhos.

In [6]:
TOPIC_NAMES = {
    "algoritmos aprendizado": "Algoritmos, otimização e aprendizado de máquina",
    "sistemas controle": "Controle, robótica e sistemas embarcados",
    "programas linguagens": "Programação e sistemas computacionais",
    "software requisitos": "Engenharia de software e gestão",
    "funções equações": "Cálculo, análise e equações",
    "espaços lineares": "Álgebra, geometria e topologia",
    "modelos testes": "Probabilidade e estatística",
    "circuitos corrente": "Circuitos, eletricidade e sistemas digitais",
    "momento mecânica": "Física, mecânica e termodinâmica",
    "química reações": "Química e biologia",
    "ensino matemática": "Ensino de matemática e ciências",
    "dados mineração": "Bases de dados, mineração e ciência de dados",
    "projeto empresa": "Estágios, projetos e metodologia científica",
    "educação ciências": "Educação",
    "computação oferecimento": "Seminários e tópicos especiais",
    "ondas quântica": "Ondas e telecomunicações",
    "atividades iniciação": "Atividades complementares e tutoria",
    "extensão discente": "Extensão universitária",
    "desenho técnico": "Desenho técnico, materiais e fabricação",
    "difração óptica": "Óptica e fotônica",
    "redes móveis": "Redes de computadores",
    "produção sistemas": "Produção e sistemas operacionais",
    "imagens processamento": "Processamento de imagens e visão",
    "microcontroladores projetos": "Microcontroladores e arquitetura"
}

topic_name = [TOPIC_NAMES.get(k, " · ".join(w for w, _ in words[t][:3])) for t, k in enumerate(keys)]
C = np.array([E[topic == t].mean(axis=0) for t in range(T)])
C /= np.linalg.norm(C, axis=1, keepdims=True)
topic_order = [int(t) for t in leaves_list(linkage(pdist(C, "cosine"), "average", optimal_ordering=True))]
missing = [k for k in keys if k not in TOPIC_NAMES]
print("tópicos sem nome:", missing or "nenhum")

dictionary = pd.DataFrame([{"topic": t, "topic_name": topic_name[t], "rank": r + 1,
                            "word": w, "weight": round(v, 4)} for t in range(T) for r, (w, v) in enumerate(words[t])])
(dictionary[dictionary["rank"] <= 10].groupby(["topic", "topic_name"])["word"].apply(", ".join)
 .reset_index().rename(columns={"word": "palavras de maior peso (c-TF-IDF)"}))

tópicos sem nome: nenhum


,topic,topic_name,palavras de maior peso (c-TF-IDF)
0,0,"Algoritmos, otimização e aprendizado de máquina","algoritmos, aprendizado, árvores, busca, objet..."
1,1,"Controle, robótica e sistemas embarcados","sistemas, controle, robôs, embarcados, navegaç..."
2,2,Programação e sistemas computacionais,"programas, linguagens, programação, computação..."
3,3,Engenharia de software e gestão,"software, requisitos, gerenciamento, sistemas,..."
4,4,"Cálculo, análise e equações","funções, equações, séries, números, teorema, m..."
5,5,"Álgebra, geometria e topologia","espaços, lineares, teorema, matrizes, grupos, ..."
6,6,Probabilidade e estatística,"modelos, testes, regressão, estatística, variá..."
7,7,"Circuitos, eletricidade e sistemas digitais","circuitos, corrente, elétricos, transistores, ..."
8,8,"Física, mecânica e termodinâmica","momento, mecânica, movimento, energia, calor, ..."
9,9,Química e biologia,"química, reações, ensino, estágio, escola, quí..."


## **6. Rede e mapa**

Cada disciplina liga-se às quatro vizinhas com $s(a,b)$ acima do percentil 85, e a vizinha mais próxima é mantida mesmo abaixo do limiar. A posição vem de um UMAP bidimensional e serve para leitura espacial, não como distância exata. Textos idênticos são projetados uma única vez e compartilham a coordenada (`unique=True`). Sem isso, o UMAP os isola como um componente desconexo. As coordenadas são giradas para que o eixo de maior variância fique na horizontal e reescaladas por um único fator, pois normalizar cada eixo separadamente deformaria a projeção. A rede será exportada em `data/processed/similarity_edges.csv`.

In [7]:
edges = set()
for i in range(n):
    for rank, j in enumerate(np.argsort(-S[i])[:K]):
        if rank == 0 or S[i, j] >= FLOOR:
            edges.add((min(i, int(j)), max(i, int(j))))
edges = sorted(edges)
xy = UMAP(n_neighbors=12, n_components=2, min_dist=0.25, metric="cosine", unique=True, random_state=SEED).fit_transform(E)
xy = xy - xy.mean(axis=0)
xy = xy @ np.linalg.svd(xy, full_matrices=False)[2].T
xy = (xy - xy.min(axis=0)) / np.ptp(xy, axis=0).max()
degree = Counter([a for a, _ in edges] + [b for _, b in edges])
print(f"{len(edges)} ligações · grau mediano {np.median([degree[i] for i in range(n)]):.0f} · altura/largura do mapa {np.ptp(xy[:, 1]):.2f}")

1499 ligações · grau mediano 5 · altura/largura do mapa 0.77


## **7. Medidas para as perguntas**

As três perguntas usam os mesmos identificadores de disciplina e tópico.

### **7.1. Q1 · Tópicos e departamentos**

A informação mútua normalizada resume a associação entre tópico $T$ e departamento $D$: $\operatorname{NMI}(T,D)\in[0,1]$. Zero indica independência, e um indica partições idênticas. Como a NMI cresce com o número de grupos mesmo ao acaso, também é informada a versão ajustada (AMI), cujo valor esperado sob independência é zero. O departamento é a sigla do código representante, e códigos numéricos recebem a sigla da unidade.

In [8]:
prefix = subjects["prefix"]
nmi = normalized_mutual_info_score(prefix, topic)
ami = adjusted_mutual_info_score(prefix, topic)
print(f"Q1 · informação mútua tópico × departamento: NMI {nmi:.2f} · AMI {ami:.2f}")
display(pd.crosstab(pd.Series([topic_name[t] for t in topic], name="tópico"), prefix.rename("prefixo")))

Q1 · informação mútua tópico × departamento: NMI 0.47 · AMI 0.40


prefixo,EESC,EESC-ICMC,IAU,ICMC,IFSC,IQSC,LCE,RAD,SAA,SCC,SEL,SEM,SEP,SET,SHS,SLC,SMA,SME,SMM,SSC
tópico,,,,,,,,,,,,,,,,,,,,
"Algoritmos, otimização e aprendizado de máquina",0,0,0,0,0,0,0,0,0,24,0,0,0,0,0,0,0,9,0,4
Atividades complementares e tutoria,7,1,0,0,0,0,3,0,0,0,0,0,0,0,0,1,0,0,0,0
"Bases de dados, mineração e ciência de dados",0,0,0,2,0,0,0,1,0,13,0,0,0,0,0,0,0,7,0,2
"Circuitos, eletricidade e sistemas digitais",0,0,0,0,3,0,0,0,0,0,19,0,0,0,0,2,0,0,0,6
"Controle, robótica e sistemas embarcados",1,0,0,1,0,0,0,0,3,1,16,3,0,0,0,2,0,0,1,7
"Cálculo, análise e equações",0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,3,19,8,0,0
"Desenho técnico, materiais e fabricação",0,0,1,0,0,0,0,0,0,0,0,6,2,0,0,0,0,0,2,0
Educação,0,0,3,0,1,0,0,0,0,0,0,0,0,0,0,9,4,0,0,0
Engenharia de software e gestão,0,0,0,1,0,0,0,1,0,1,0,0,6,0,0,0,0,1,0,23


### **7.2. Q3 · Grades por curso**

`data/raw/curriculum.csv` é reescrito com os identificadores das disciplinas únicas. Em duplicidades, obrigatórias prevalecem sobre optativas e fica o semestre mais cedo. A carga horária é a da linha da grade, pois o mesmo conteúdo pode ter cargas distintas em cada curso.

In [9]:
nn = S.argmax(axis=1)
nn_sim = S[np.arange(n), nn]
curriculum = pd.read_csv(RAW_DATA / "curriculum.csv")
course_names = pd.read_csv(RAW_DATA / "courses.csv").set_index("course_id")["course_name"]
code_to_subject = {c: k for k, codes in enumerate(subjects["codes"]) for c in codes.split()}
curriculum["k"] = curriculum["code"].map(code_to_subject)
curriculum = curriculum.dropna(subset=["k"]).astype({"k": int})
KIND = {"obrigatória": 0, "optativa eletiva": 1, "optativa livre": 2}
grade = (curriculum.assign(kind_rank=curriculum["kind"].map(KIND).fillna(2))
         .sort_values(["course_id", "k", "kind_rank", "semester"])
         .drop_duplicates(["course_id", "k"]))
grade = grade[grade.groupby("course_id")["k"].transform("size") >= 10].copy()
grade["mandatory"] = grade["kind_rank"].eq(0)
grade["hours"] = grade["workload_hours"].fillna(0)
grade["topic"] = topic[grade["k"]]
print(f"{grade['course_id'].nunique()} cursos, {len(grade)} disciplinas nas grades "
      f"({int(grade['mandatory'].sum())} obrigatórias)")
grade.groupby("course_id").agg(disciplinas=("k", "size"), obrigatórias=("mandatory", "sum"), semestres=("semester", "max"))

12 cursos, 1061 disciplinas nas grades (502 obrigatórias)


,disciplinas,obrigatórias,semestres
course_id,,,
55030-200,84,26,8
55030-300,78,32,8
55041-0,136,45,10
55051-4,83,43,8
55060-0,82,30,8
55071-4,75,32,8
55090-0,104,30,8
90011-104,44,40,8
90011-204,59,55,8


### **7.3. Q3 · Rede de requisitos**

Os requisitos são filtrados por curso e pelas disciplinas presentes na respectiva grade. Para Matemática, bacharelado e licenciatura são separados. Indicações de conjunto não entram no grafo: pedem que duas disciplinas sejam cursadas juntas e não estabelecem precedência. A saída mede pares, cadeia mais longa e alcance de cada requisito. A semelhança dos pares é comparada com todos os pares e com os pares de disciplinas da mesma grade.

In [10]:
requisites_all = pd.read_csv(RAW_DATA / "requisites.csv", dtype={"course_code": str})
HABILITATION = {"55030-200": "Bacharelado", "55030-300": "Licenciatura"}
course_reqs = []
for cid, g in grade.groupby("course_id"):
    rows = requisites_all[(requisites_all["course_code"] == cid.split("-")[0]) & (requisites_all["relation"] != "indicação de conjunto")]
    if cid in HABILITATION:
        rows = rows[rows["course_name"].str.contains(HABILITATION[cid])]
    in_grade = set(g["k"])
    for r in rows.itertuples():
        a, b = code_to_subject.get(r.requisite), code_to_subject.get(r.code)
        if a is None or b is None or a == b or a not in in_grade or b not in in_grade:
            continue
        course_reqs.append({"course_id": cid, "a": a, "b": b, "strong": r.relation == "requisito"})
course_reqs = (pd.DataFrame(course_reqs).groupby(["course_id", "a", "b"])["strong"].any().reset_index())
course_reqs["similarity"] = [round(float(S[a, b]), 3) for a, b in zip(course_reqs["a"], course_reqs["b"])]

summary = []
for cid, g in course_reqs.groupby("course_id"):
    G = nx.DiGraph(list(zip(g["a"], g["b"])))
    for c in list(nx.simple_cycles(G)):
        if G.has_edge(c[-1], c[0]):
            G.remove_edge(c[-1], c[0])
    longest = nx.dag_longest_path(G)
    hub = max(G.nodes, key=lambda v: len(nx.descendants(G, v)))
    summary.append({"curso": course_names.get(cid, cid), "pares": len(g), "disciplinas com requisito": g["b"].nunique(),
                    "cadeia mais longa (etapas)": len(longest) - 1,
                    "cadeia": " → ".join(subjects.at[v, "name"] for v in longest),
                    "mais disciplinas dependem de": f"{subjects.at[hub, 'name']} ({len(nx.descendants(G, hub))})"})
same_grade = np.concatenate([S[np.ix_(k, k)][np.triu_indices(len(k), 1)] for k in (g["k"].to_numpy() for _, g in grade.groupby("course_id"))])
distinct = course_reqs.drop_duplicates(["a", "b"])["similarity"]
print(f"Q3 · {len(course_reqs)} pares requisito → disciplina dentro das grades ({len(distinct)} distintos) · semelhança mediana "
      f"{course_reqs['similarity'].median():.2f} (distintos {distinct.median():.2f}) vs. mesma grade {np.median(same_grade):.2f} "
      f"e todos os pares {np.median(all_pairs):.2f}")
pd.DataFrame(summary)

Q3 · 897 pares requisito → disciplina dentro das grades (576 distintos) · semelhança mediana 0.64 (distintos 0.64) vs. mesma grade 0.56 e todos os pares 0.54


,curso,pares,disciplinas com requisito,cadeia mais longa (etapas),cadeia,mais disciplinas dependem de
0,Bacharelado em Matemática,76,56,4,Introdução à Programação de Computadores → Int...,Cálculo I (20)
1,Licenciatura em Matemática,62,45,4,Cálculo I → Cálculo II → Espaços Métricos → To...,Cálculo I (19)
2,Bacharelado em Ciências de Computação,107,88,6,Introdução à Lógica Digital → Sistemas Digitai...,Algoritmos e Estruturas de Dados I (51)
3,Bacharelado em Sistemas de Informação,63,54,5,Introdução à Ciência de Computação I → Algorit...,Introdução à Ciência de Computação I (40)
4,Bacharelado em Matemática Aplicada e Computaçã...,77,59,4,Cálculo I → Física I → Física II → Fundamentos...,Cálculo I (31)
5,Bacharelado em Estatística e Ciência de Dados,74,54,4,Introdução à Ciência de Computação I → Estrutu...,Cálculo I (38)
6,Bacharelado em Ciência de Dados,100,79,5,Introdução à Ciência de Computação I → Estrutu...,Introdução à Ciência de Computação I (45)
7,Licenciatura em Ciências Exatas - Núcleo Geral,37,22,5,Elementos de Matemática → Cálculo I → Física I...,Elementos de Matemática (14)
8,Licenciatura em Ciências Exatas - Habilitação ...,63,35,7,Elementos de Matemática → Cálculo I → Física I...,Elementos de Matemática (27)
9,Licenciatura em Ciências Exatas - Habilitação ...,54,37,7,Elementos de Matemática → Cálculo I → Física I...,Química 1 (26)


### **7.4. Carga horária por tópico**

Para cada curso $c$ e tópico $t$, soma-se $H_{c,t}=\sum_{d\in(c,t)}h_d$. A tabela mantém cursos nas colunas e tópicos nas linhas. A medida prepara a composição dos cursos da próxima entrega e não aparece na interface atual.

In [11]:
course_topic = (grade.assign(course=grade["course_id"].map(course_names), topic_name=grade["topic"].map(lambda t: topic_name[t]))
                .pivot_table(index="topic_name", columns="course", values="hours", aggfunc="sum", fill_value=0))
print("Horas por tópico em cada curso (todas as disciplinas):")
course_topic.astype(int)

Horas por tópico em cada curso (todas as disciplinas):


course,Bacharelado em Ciência de Dados,Bacharelado em Ciências de Computação,Bacharelado em Estatística e Ciência de Dados,Bacharelado em Matemática,Bacharelado em Matemática Aplicada e Computação Científica,Bacharelado em Sistemas de Informação,Engenharia de Computação,Licenciatura em Ciências Exatas - Habilitação em Física,Licenciatura em Ciências Exatas - Habilitação em Matemática,Licenciatura em Ciências Exatas - Habilitação em Química,Licenciatura em Ciências Exatas - Núcleo Geral,Licenciatura em Matemática
topic_name,,,,,,,,,,,,
"Algoritmos, otimização e aprendizado de máquina",1725,1800,645,1155,1305,1290,1380,0,0,0,0,450
Atividades complementares e tutoria,0,0,0,0,0,0,420,135,135,135,135,0
"Bases de dados, mineração e ciência de dados",1695,1035,1260,450,540,540,645,0,0,0,0,0
"Circuitos, eletricidade e sistemas digitais",90,330,0,60,60,60,600,150,0,0,0,60
"Controle, robótica e sistemas embarcados",120,795,0,0,0,120,1890,150,120,60,60,75
"Cálculo, análise e equações",660,360,600,1410,1170,150,360,180,420,180,180,1530
"Desenho técnico, materiais e fabricação",0,0,0,0,0,0,210,0,0,0,0,0
Educação,0,30,360,450,0,0,30,660,660,660,660,840
Engenharia de software e gestão,405,1185,105,0,105,1200,945,0,0,0,0,45


## **8. Exportação**

A etapa final grava as medidas em `data/processed/` e reúne o conteúdo consumido pelo site em `html/data/disciplinas.js`.

| arquivo | conteúdo |
|---|---|
| `subject_topics.csv` | tópico, posição e vizinha de cada disciplina |
| `topics.csv`, `topic_dictionary.csv` | nomes, tamanhos e palavras |
| `similarity_edges.csv` | rede de semelhança |
| `course_requisites.csv` | requisitos por curso |
| `course_topic_hours.csv` | carga por tópico e curso |
| `html/data/disciplinas.js` | pacote completo da visualização |

In [12]:
nodes = []
for k, row in subjects.iterrows():
    nodes.append({
        "id": row["subject_id"], "codes": row["codes"].split(), "name": row["name"], "dept": prefix[k],
        "topic": int(topic[k]),
        "x": round(float(xy[k, 0]), 4), "y": round(float(xy[k, 1]), 4),
        "nn": int(nn[k]), "nnSim": round(float(nn_sim[k]), 3),
        "credits": None if pd.isna(row["credits"]) else int(row["credits"]),
        "hours": None if pd.isna(row["workload_hours"]) else int(row["workload_hours"]),
        "period": None if pd.isna(row["ideal_period"]) else int(row["ideal_period"]),
        "courses": int(row["n_courses"]), "teachers": row["teachers"],
        "ementa": row["ementa"], "objectives": row["objectives"], "program": row["program"],
        "near": [[int(j), round(float(S[k, j]), 3)] for j in np.argsort(-S[k])[:8]],
    })
topics_out = [{"id": t, "key": keys[t], "name": topic_name[t], "size": int((topic == t).sum()),
               "words": [[w, round(v, 4)] for w, v in words[t]],
               "depts": dict(Counter(prefix[topic == t]).most_common())} for t in range(T)]
payload = {
    "meta": {"collectedAt": collection["collected_at_utc"], "pages": int(collection["disciplines_total"]), "subjects": n,
             "topics": T, "model": MODEL, "k": K, "floor": round(FLOOR, 3), "nmi": round(float(nmi), 3), "ami": round(float(ami), 3),
             "medianAll": round(float(np.median(all_pairs)), 3), "p90All": round(P90, 3)},
    "topics": topics_out, "topicOrder": topic_order, "nodes": nodes,
    "edges": [[a, b, round(float(S[a, b]), 3)] for a, b in edges],
    # semelhança de cada disciplina com todas as outras, em centésimos (usada no beeswarm da Q2)
    "sim": [[int(round(max(0.0, float(v)) * 100)) for v in row] for row in S],
    "courses": [{"id": cid, "name": course_names.get(cid, cid),
                 "items": [[int(r.k), int(r.mandatory), int(r.semester), int(r.kind_rank)] for r in g.itertuples()],
                 "reqs": [[int(r.a), int(r.b), int(r.strong)] for r in course_reqs[course_reqs["course_id"] == cid].itertuples()]}
                for cid, g in grade.groupby("course_id")],
}
SITE_DATA.mkdir(parents=True, exist_ok=True)
(SITE_DATA / "disciplinas.js").write_text("window.DATA=" + json.dumps(payload, ensure_ascii=False, separators=(",", ":")) + ";\n", encoding="utf-8")

outputs = {
    "subject_topics.csv": pd.DataFrame({"subject_id": subjects["subject_id"], "name": subjects["name"], "topic": topic,
                                        "topic_name": [topic_name[t] for t in topic],
                                        "x": xy[:, 0].round(4), "y": xy[:, 1].round(4),
                                        "nearest": subjects["subject_id"].to_numpy()[nn], "nearest_similarity": nn_sim.round(3)}),
    "topics.csv": pd.DataFrame([{"topic": t["id"], "name": t["name"], "size": t["size"],
                                 "words": ", ".join(w for w, _ in t["words"][:15])} for t in topics_out]),
    "topic_dictionary.csv": dictionary,
    "similarity_edges.csv": pd.DataFrame([{"source": subjects.at[a, "subject_id"], "target": subjects.at[b, "subject_id"],
                                           "similarity": round(float(S[a, b]), 3)} for a, b in edges]),
    "course_requisites.csv": course_reqs.assign(course=course_reqs["course_id"].map(course_names),
                                                requisite=subjects["subject_id"].to_numpy()[course_reqs["a"]],
                                                discipline=subjects["subject_id"].to_numpy()[course_reqs["b"]]).drop(columns=["a", "b"]),
    "course_topic_hours.csv": course_topic.reset_index(),
}
for name, frame in outputs.items():
    frame.to_csv(PROCESSED / name, index=False)
artifacts = [{"artifact": name, "rows": len(frame), "directory": "data/processed"} for name, frame in outputs.items()]
artifacts.append({"artifact": "disciplinas.js", "rows": n, "directory": "html/data"})
pd.DataFrame(artifacts)

,artifact,rows,directory
0,subject_topics.csv,528,data/processed
1,topics.csv,24,data/processed
2,topic_dictionary.csv,720,data/processed
3,similarity_edges.csv,1499,data/processed
4,course_requisites.csv,897,data/processed
5,course_topic_hours.csv,24,data/processed
6,disciplinas.js,528,html/data
